# ECG-FM Stage 2 Fine-tuning
**ECG-FM foundation model (90M params) — full encoder fine-tune on PTB-XL**

### Before running:
1. In Colab menu: **Runtime → Change runtime type → T4 GPU**
2. Upload these two files to your Google Drive under `My Drive/EKG_models/`:
   - `mimic_iv_ecg_physionet_pretrained.pt` (~365 MB) — from your local `models/ecgfm/`
   - `ecgfm_stage1.pt` (~365 MB) — from your local `models/`

### Why this notebook downloads to Google Drive (important)
Colab free tier disconnects after ~90 min of inactivity or when the runtime hits its limit.
PTB-XL (500Hz) is ~6 GB — it can take 20–60 min depending on PhysioNet speed.
**Files in `/content/` are deleted on disconnect. Files in Drive persist.**
Cell 4 downloads to Drive and uses `-c` (resume). If you get disconnected:
1. Reconnect runtime
2. Re-run Cells 1–4 — the download will resume from where it stopped
3. Then run Cells 5–7

### Why we use 500Hz (not 100Hz)
ECG-FM was pretrained on 500Hz MIMIC-IV signals. At 100Hz the transformer
receives only ~3 tokens instead of ~15 — performance would collapse.
Do NOT switch to `records100`.

In [ ]:
# Cell 1 — Check GPU and RAM
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU — training will be extremely slow. Change runtime type to T4.')

import psutil
print('System RAM:', round(psutil.virtual_memory().total / 1e9, 1), 'GB')
print('RAM available:', round(psutil.virtual_memory().available / 1e9, 1), 'GB')

In [ ]:
# Cell 2 — Mount Google Drive and verify model files
from google.colab import drive
drive.mount('/content/drive')

import os
MODEL_DIR = '/content/drive/MyDrive/EKG_models'
assert os.path.exists(f'{MODEL_DIR}/mimic_iv_ecg_physionet_pretrained.pt'), \
    f'Missing: upload mimic_iv_ecg_physionet_pretrained.pt to {MODEL_DIR}/'
assert os.path.exists(f'{MODEL_DIR}/ecgfm_stage1.pt'), \
    f'Missing: upload ecgfm_stage1.pt to {MODEL_DIR}/'
print('Drive model files found OK')

# PTB-XL will be downloaded here (persists across sessions)
DRIVE_PTBXL = f'{MODEL_DIR}/ptbxl'
os.makedirs(DRIVE_PTBXL, exist_ok=True)
print(f'PTB-XL destination: {DRIVE_PTBXL}')

In [ ]:
# Cell 3 — Clone repo and install dependencies
import os
if not os.path.exists('EKG'):
    !git clone https://github.com/shlomih/EKG.git
else:
    print('Repo already cloned')

%cd EKG
!pip install -q wfdb scipy scikit-learn
print('Setup done')

In [ ]:
# Cell 4 — Download PTB-XL 500Hz signals to Google Drive
#
# KEY DESIGN:
#   - Downloads to Drive (not /content/) so files survive runtime disconnection
#   - Uses wget -nc (no-clobber) — skips files already on Drive, downloads only new ones
#   - Uses records500 (500Hz) — required by ECG-FM architecture
#   - After download, symlinks Drive path → local path expected by the training script
#
# If your runtime disconnects during download:
#   1. Reconnect and re-run Cells 1-4
#   2. Cell 4 skips all files already on Drive — only downloads what's missing
#
# NOTE: Do NOT use -N and -c together — they are incompatible flags.
#   wget ignores -N when -c is present and re-downloads everything from scratch.
#   Use -nc (--no-clobber) instead: it simply skips existing files.

import os
from pathlib import Path
import pandas as pd

DRIVE_PTBXL = '/content/drive/MyDrive/EKG_models/ptbxl'
LOCAL_LINK  = 'ekg_datasets/ptbxl'
os.makedirs(DRIVE_PTBXL, exist_ok=True)
os.makedirs('ekg_datasets', exist_ok=True)

# ── Step 1: metadata CSV (tiny, download always) ─────────────────────────
for fname in ['ptbxl_database.csv', 'scp_statements.csv']:
    dest = f'{DRIVE_PTBXL}/{fname}'
    if not os.path.exists(dest):
        print(f'Downloading {fname}...')
        !wget -q -nc -P {DRIVE_PTBXL} \
            https://physionet.org/files/ptb-xl/1.0.3/{fname}
    else:
        print(f'{fname} already on Drive')

# ── Step 2: 500Hz signal files (skip existing) ───────────────────────────
rec_dir = Path(f'{DRIVE_PTBXL}/records500')
n_dat = len(list(rec_dir.rglob('*.dat'))) if rec_dir.exists() else 0
print(f'\nSignal files on Drive: {n_dat} / 21837')

if n_dat < 21000:
    print('Downloading records500 to Drive (skipping files already present)...')
    print('(Safe to interrupt — re-run this cell to continue)\n')
    # -r  recursive   -nc  no-clobber (skip existing files)
    # -np no parent   -nH  no hostname prefix   --cut-dirs=3  strip /files/ptb-xl/1.0.3/
    !wget -r -nc -np -nH --cut-dirs=3 \
        -P {DRIVE_PTBXL} \
        https://physionet.org/files/ptb-xl/1.0.3/records500/
    n_dat = len(list(rec_dir.rglob('*.dat')))
    print(f'After download: {n_dat} .dat files on Drive')
else:
    print('All records already on Drive — skipping download')

# ── Step 3: symlink Drive → local path ───────────────────────────────────
# ecgfm_finetune.py expects 'ekg_datasets/ptbxl' as the dataset root
if os.path.islink(LOCAL_LINK):
    os.remove(LOCAL_LINK)
elif os.path.isdir(LOCAL_LINK):
    import shutil; shutil.rmtree(LOCAL_LINK)
os.symlink(os.path.abspath(DRIVE_PTBXL), LOCAL_LINK)
print(f'\nLinked: {LOCAL_LINK} → {os.path.realpath(LOCAL_LINK)}')

# ── Verify ───────────────────────────────────────────────────────────────
df = pd.read_csv(f'{DRIVE_PTBXL}/ptbxl_database.csv')
print(f'PTB-XL CSV: {len(df)} records')
print(f'Signal files: {n_dat}')
if n_dat < 21000:
    print(f'\nWARNING: Only {n_dat}/21837 signal files downloaded.')
    print('Re-run this cell to continue. Training will fail if signals are missing.')
else:
    print('\nReady for training.')

In [ ]:
# Cell 5 — Copy model files from Google Drive into the project
import shutil, os

os.makedirs('models/ecgfm', exist_ok=True)

shutil.copy(f'{MODEL_DIR}/mimic_iv_ecg_physionet_pretrained.pt',
            'models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt')
shutil.copy(f'{MODEL_DIR}/ecgfm_stage1.pt',
            'models/ecgfm_stage1.pt')

print('Model files copied:')
print(' ', round(os.path.getsize('models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt')/1e6), 'MB — pretrained encoder')
print(' ', round(os.path.getsize('models/ecgfm_stage1.pt')/1e6), 'MB — stage 1 checkpoint')

In [ ]:
# Cell 6 — Run Stage 2 fine-tuning
#
# What happens:
#   - Preloads train+val signals into RAM (~4 GB) — test signals deferred until after training
#   - Unfreezes full 90M param encoder on GPU (auto-detected)
#   - Training: ~20 min/epoch, ~3 hours total with early stopping (patience=8)
#   - Saves checkpoint to models/ecgfm_stage2.pt at every validation improvement
#
# If you get CUDA out-of-memory error:
#   Change --batch_s2 8  to  --batch_s2 4  below
#
# Note: torch.set_num_threads(2) is already set in ecgfm_finetune.py (correct for Colab's 2 cores)

!python -u ecgfm_finetune.py --stage 2 --batch_s2 8

In [ ]:
# Cell 7 — Save result back to Google Drive
import os, shutil, torch

if os.path.exists('models/ecgfm_stage2.pt'):
    dest = f'{MODEL_DIR}/ecgfm_stage2.pt'
    shutil.copy('models/ecgfm_stage2.pt', dest)
    print(f'Saved ecgfm_stage2.pt ({round(os.path.getsize(dest)/1e6)} MB) to Google Drive')

    ckpt = torch.load('models/ecgfm_stage2.pt', map_location='cpu', weights_only=False)
    tm   = ckpt.get('test_metrics', {})
    print(f'\nFinal test results:')
    print(f'  Accuracy  : {tm.get("acc", 0):.1%}')
    print(f'  HYP F1    : {tm.get("hyp_f1", 0):.3f}')
    print(f'  Macro F1  : {tm.get("macro_f1", 0):.3f}')
else:
    print('WARNING: ecgfm_stage2.pt not found — training may not have completed')

## After training

Download `ecgfm_stage2.pt` from your Google Drive (`EKG_models/ecgfm_stage2.pt`) to your local `models/` folder.

### What to expect
| Metric | Stage 1 baseline | Target |
|---|---|---|
| HYP F1 | 0.375 test | > 0.50 |
| Macro F1 | 0.492 test | > 0.70 |

If ECG-FM Stage 2 beats the current multi-label CNN (MacroF1=0.699, MacroAUROC=0.972), it can replace the CNN backbone.

See `DECISIONS.md` → D-006 and D-007 for context.

### If the runtime disconnected during Cell 4
PTB-XL is saved to Drive — it survived. Reconnect and re-run Cells 1–4.
Cell 4 uses `wget -nc` (no-clobber) so it skips files already on Drive and only downloads what's missing.